# មេរៀន 01 - ការណែនាំអោយបានស្គាល់អេហ្សិន AI

សូមស្វាគមន៍មកកាន់មេរៀនដំបូងក្នុងវគ្គ **AI Agents សម្រាប់អ្នកចាប់ផ្តើម**!

**អេហ្សិន AI** គឺជាប្រограмមួយដែលប្រើម៉ូដែលភាសាធំនឹង (LLM) ជាជនឧបករណ៍សរសេរ និងអាចអនុវត្តន៍ *សកម្មភាព* នៅក្នុងលោកពិតប្រាកដ — ហៅ API, ស្វែងរកទិន្នន័យក្នុងមូលដ្ឋានទិន្នន័យ ឬ ប្រតិបត្តិការកូដ — ដើម្បីសម្រេចគោលបំណងមួយវិញមកបើកដោយអ្នកប្រើ។

នៅក្នុងសៀវភៅតារាងនេះអ្នកនឹងបង្កើតអេហ្សិនដំបូងរបស់អ្នក: **អេហ្សិនធ្វើដំណើរ** ដែលផ្តល់អនុសាសន៍កន្លែងធ្វើដំណើរ។ នៅក្នុងដំណើរនេះអ្នកនឹងរៀនពីរបៀប:

1. តភ្ជាប់ទៅកាន់សេវាកម្ម Microsoft Foundry Agent តាមរយៈ **Microsoft Agent Framework**។
2. ផ្តល់ឧបករណ៍មួយ​អោយអេហ្សិន — ជាផ្នែកមុខងារ Python ដែលវាអាចហៅបាន។
3. រត់អេហ្សិន ហើយពិនិត្យមើលចម្លើយរបស់វា។
4. បញ្ជូនចម្លើយរបស់អេហ្សិនជាច_tokens​តាមធាតុមួយៗ។


## ការតំឡើង

មុនពេលរត់កំណត់ចំណាំនេះ សូមប្រាកដថាអ្នកមាន៖

1. **គម្រោង Microsoft Foundry មួយ** ដែលមានម៉ូដែលជជែកបានដាក់អនុវត្ត (ឧ. `gpt-5-mini`)។
2. **បានចូលប្រើជាមួយ Azure CLI** — រត់ `az login` នៅក្នុងបន្ទាត់ពាក្យបញ្ជារបស់អ្នក។
3. **កំណត់អថេរបរិស្ថានដែលត្រូវការជា៖**
   - `AZURE_AI_PROJECT_ENDPOINT` — ចំណុចផ្ដួចគម្រោង Microsoft Foundry របស់អ្នក។
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — ឈ្មោះម៉ូដែលដែលបានដាក់អនុវត្តរបស់អ្នក។

កោសិកាក្រោមនេះដំឡើងកញ្ចប់ Python ដែលអ្នកត្រូវការ។


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## ការបង្កើតភ្នាក់ងារដំបូងរបស់អ្នក

ភ្នាក់ងារត្រូវការពីរពីរប្រភេទ:

- **ការណែនាំ** ដែលប្រាប់វា *អ្នកជាអ្នកណា* និង *របៀបបង្ហាញខ្លួន* (ជាការផ្តល់ប្រព័ន្ធ).
- **ឧបករណ៍** — លំនាំ Python ដែលត្រូវបានតុបតែងជាមួយ `@tool` ដែលភ្នាក់ងារអាចហៅដើម្បីទទួលព័ត៌មាន ឬអនុវត្តសកម្មភាព។

ខាងក្រោមនេះយើងបានកំណត់ឧបករណ៍មួយសាមញ្ញដែលបង្រ្កាបបញ្ជីទីកន្លែងទេសចរណ៍ពេញនិយម។ ភ្នាក់ងារនឹងប្រើឧបករណ៍នេះនៅពេលអ្នកប្រើស្នើសុំការផ្តល់អនុសាសន៍ដំណើរកម្សាន្ត។


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## ផ្ទុកចេញជាប្រព័ន្ធផ្សាយបន្តផ្ទាល់

សម្រាប់បទពិសោធន៍អន្តរកម្មបន្ថែម អ្នកអាច **ផ្ទុកចេញជាប្រព័ន្ធផ្សាយបន្តផ្ទាល់** នូវការឆ្លើយតបរបស់ភ្នាក់ងារ។ ផ្ទុយពីការរង់ចាំការឆ្លើយតបពេញលេញ ភ្នាក់ងារនឹងបញ្ចេញអត្ថបទជាគន្លងតូចៗ កាលពីពួកវាត្រូវបានបង្កើត។ វាមានប្រយោជន៍ពិសេសនៅក្នុងចំណុចជជែក ដែលអ្នកចង់បង្ហាញលទ្ធផលនៅពេលវេលាជាក់ស្តែង។


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## សរុប

ក្នុងមេរៀននេះ អ្នកបានរៀនពីរបៀបធ្វើដូចតទៅ៖

- **បង្កើតអ្នកផ្គត់ផ្គង់** ដែលភ្ជាប់ទៅនឹងសេវាកម្ម Microsoft Foundry Agent តាមរយៈ `FoundryChatClient`។
- **កំណត់ឧបករណ៍** ដោយប្រើ `@tool` decorator ដើម្បីឲ្យភ្នាក់ងារអាចហៅមុខងារ Python របស់អ្នកបាន។
- **ដំណើរការភ្នាក់ងារ** ជាមួយសារ​របស់អ្នកប្រើ ហើយបោះពុម្ពចម្លើយ​របស់វា។
- **ផ្សាយចម្លើយបន្ត** សម្រាប់លទ្ធផលពេលវេលាស្រប។

ក្នុងមេរៀនបន្ទាប់ ពួកយើងនឹងស្រាវជ្រាវស៊្រង agentic ជ្រាលជ្រៅ និងរៀនពីរបៀបផ្តល់ឱ្យភ្នាក់ងារនូវឧបករណ៍មានអំណាចបន្ថែម និងសមត្ថភាពយល់ព្រមជាច្រើនជំហាន។


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
